In [1]:
import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np
from tqdm.auto import tqdm
import os

# Register the progress_apply method with pandas
tqdm.pandas(desc="Applying function to DataFrame")

In [2]:
# Define the paths to your Xenium output files
output_dir = "/home/xilab/shared/xenium-spatial-202501/20250122_spatial_transcriptome_fetal_and_tumor/output-XETG00129__0041503__Region_1__20250115__051412/"
cells_path = os.path.join(output_dir, "cells.parquet") # Or "cells.csv.gz"
cells_features_path = os.path.join(output_dir, "cell_feature_matrix.h5")
features_path = os.path.join(output_dir, "transcripts.parquet") # Or "cells.csv.gz"

In [3]:
cells_df = pd.read_parquet(cells_path)

In [4]:
cells_df.head()

,cell_id,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,nucleus_count,segmentation_method
0,aaaadeae-1,1377.188232,3372.415283,490,0,0,0,0,0,490,48.317189,NaN,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
1,aaaafjeh-1,1365.575684,3396.847656,383,0,0,0,0,0,383,39.015001,19.868751,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
2,aaaajmmn-1,1368.786865,3392.293945,1360,0,0,0,0,0,1360,115.464535,45.472345,3,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
3,aaaamacm-1,1375.199097,3378.217041,1035,0,0,0,0,1,1036,107.833129,30.932032,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
4,aaaanige-1,1351.775024,3411.712402,141,0,0,0,0,0,141,57.032346,NaN,0,Segmented by boundary stain (ATP1A1+CD45+E-Cad...


In [5]:
features_df = pd.read_parquet(features_path)

In [6]:
features_df.head()

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance,codeword_index,codeword_category,is_gene
0,281741264722161,UNASSIGNED,0,Fbxo41,199.562500,1494.718750,16.500000,40.00,D2,161.250000,9981,predesigned_gene,True
1,281741264744849,UNASSIGNED,0,Fh1,158.984375,1481.843750,16.718750,23.50,D2,118.687500,14061,predesigned_gene,True
2,281741264696055,UNASSIGNED,0,Ltb4r1,198.718750,1438.453125,16.390625,26.00,D2,153.140625,6602,predesigned_gene,True
3,281741264730838,UNASSIGNED,0,Mep1b,72.328125,1448.640625,16.421875,40.00,D2,26.734375,7937,predesigned_gene,True
4,281741264753837,npbkdmio-1,1,Sct,45.312500,1445.265625,16.765625,28.75,D2,0.000000,10691,predesigned_gene,True


# subset day 15.5 

In [7]:
EC_cells_anno_df = pd.read_csv('/data2/shared_data_backup/jiangdx/Xenium-spatial/mouse-embryo/featal-SI/default/E15.5_SI_EC_cells.csv')
EC_cells_anno_df.head()

,Unnamed: 0,orig.ident,nCount_Xenium,nFeature_Xenium,segmentation_method,nCount_BlankCodeword,nFeature_BlankCodeword,nCount_ControlCodeword,nFeature_ControlCodeword,nCount_ControlProbe,...,mnn_clusters_res_1.6,mnn_clusters_res_1.8,mnn_clusters_res_2,RCTD_spot_class,RCTD_first_type,RCTD_first_type_latest,RCTD_2_spot_class,RCTD_2_first_type,RCTD_2_second_type,cell_id
0,dacapecj-1,SeuratProject,1291,711,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,singlet,IPC-2,Enterocytes,dacapecj-1
1,dacapeoe-1,SeuratProject,880,546,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,singlet,IPC-2,Goblet cells,dacapeoe-1
2,dacbadpi-1,SeuratProject,1316,738,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,reject,Enterocytes,IPC-3,dacbadpi-1
3,dacbahao-1,SeuratProject,563,408,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,4,4,28,singlet,Epithelial cells,Epithelial cells,singlet,Enterocytes,IPC-1,dacbahao-1
4,dacbajip-1,SeuratProject,1340,742,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,1,1,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,reject,Enterocytes,IPC-2,dacbajip-1


In [8]:
EC_cells_df = cells_df[cells_df['cell_id'].isin(EC_cells_anno_df['cell_id'])]

In [9]:
EC_cells_df.shape

(21297, 14)

In [10]:
all_cells_anno_df = pd.read_csv('/data2/shared_data_backup/jiangdx/Xenium-spatial/mouse-embryo/featal-SI/default/E15.5_SI_all_cells.csv')
all_cells_anno_df.head()

,Unnamed: 0,orig.ident,nCount_Xenium,nFeature_Xenium,segmentation_method,nCount_BlankCodeword,nFeature_BlankCodeword,nCount_ControlCodeword,nFeature_ControlCodeword,nCount_ControlProbe,...,mnn_clusters_res_1.6,mnn_clusters_res_1.8,mnn_clusters_res_2,RCTD_spot_class,RCTD_first_type,RCTD_first_type_latest,RCTD_2_spot_class,RCTD_2_first_type,RCTD_2_second_type,cell_id
0,dacapecj-1,SeuratProject,1291,711,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,singlet,IPC-2,Enterocytes,dacapecj-1
1,dacapeoe-1,SeuratProject,880,546,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,singlet,IPC-2,Goblet cells,dacapeoe-1
2,dacbadpi-1,SeuratProject,1316,738,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,reject,Enterocytes,IPC-3,dacbadpi-1
3,dacbahao-1,SeuratProject,563,408,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,0,0,0,0,0,...,4,4,28,singlet,Epithelial cells,Epithelial cells,singlet,Enterocytes,IPC-1,dacbahao-1
4,dacbajip-1,SeuratProject,1340,742,Segmented by boundary stain (ATP1A1+CD45+E-Cad...,1,1,0,0,0,...,22,24,29,singlet,Epithelial cells,Epithelial cells,reject,Enterocytes,IPC-2,dacbajip-1


In [11]:
features_df = features_df[features_df['cell_id'].isin(all_cells_anno_df['cell_id'])]
features_df.head()

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance,codeword_index,codeword_category,is_gene
12428735,282716222801408,lihjilbc-1,0,A1cf,993.625000,14946.546875,15.421875,28.75,V3,0.390625,3773,predesigned_gene,True
12428736,282716222814362,lkkhpjpk-1,1,A1cf,948.890625,14993.984375,24.453125,40.00,V3,0.000000,3773,predesigned_gene,True
12428737,282716222816460,lihkagjm-1,0,A1cf,985.125000,14932.906250,17.218750,40.00,V3,0.109375,3773,predesigned_gene,True
12428738,282716224519604,lifffeip-1,1,A1cf,866.046875,14803.796875,16.703125,40.00,V3,0.000000,3773,predesigned_gene,True
12428739,282716224520901,ligfpmbp-1,1,A1cf,885.437500,14950.171875,16.765625,28.75,V3,0.000000,3773,predesigned_gene,True


In [12]:
features_df = features_df[features_df['is_gene']]
features_df['feature_name'].value_counts()

feature_name
H19        1516398
Igf2        909118
Epcam       570353
Pdia3       441388
Hnrnpa1     437446
            ...   
H1foo           92
Foxp3           92
Olig3           91
Ins1            80
Cxcl2           43
Name: count, Length: 5105, dtype: int64

In [13]:
features_df.shape

(77224951, 13)

In [14]:
print(features_df['x_location'].min())
print(features_df['x_location'].max())

print(features_df['y_location'].min())
print(features_df['y_location'].max())

831.9531
9789.75
14485.922
16023.3125


In [15]:
EC_cells_df.head()

,cell_id,x_centroid,y_centroid,transcript_counts,control_probe_counts,genomic_control_counts,control_codeword_counts,unassigned_codeword_counts,deprecated_codeword_counts,total_counts,cell_area,nucleus_area,nucleus_count,segmentation_method
143997,dacapecj-1,4089.754883,15255.265625,1291,0,0,0,0,0,1291,70.398596,39.556876,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
143998,dacapeoe-1,4076.696289,15267.713867,880,0,0,0,0,0,880,50.755627,29.622501,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
143999,dacbadpi-1,4085.388916,15264.426758,1316,0,0,0,0,0,1316,60.283596,37.253908,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
144000,dacbahao-1,4069.340332,15277.755859,563,0,0,0,0,0,563,27.229220,18.739844,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...
144001,dacbajip-1,4074.946533,15285.669922,1340,0,0,0,1,0,1341,66.018440,39.602033,1,Segmented by boundary stain (ATP1A1+CD45+E-Cad...


In [ ]:
from scipy.spatial import cKDTree

# ============================================
# 优化说明: 使用 cKDTree 进行最近邻搜索
# ============================================
# 原方法: 对每个转录本遍历所有EC细胞 O(m×n)
# 优化后: 使用KD树空间索引 O(m×log n)，加速100-1000倍
# ============================================

# 步骤1: 提取EC细胞的坐标，构建KD树索引
# cKDTree 是 scipy 提供的高性能空间索引结构，专门用于快速最近邻搜索
ec_coords = EC_cells_df[['x_centroid', 'y_centroid']].values
kdtree = cKDTree(ec_coords)

# 步骤2: 提取所有转录本的坐标（向量化）
transcript_coords = features_df[['x_location', 'y_location']].values

# 步骤3: 批量查询每个转录本到最近EC细胞的距离
# k=1 表示只查询最近的1个点
# distances: 每个转录本到最近EC细胞的距离数组
# _: 最近EC细胞的索引（这里不需要，用_忽略）
distances, _ = kdtree.query(transcript_coords, k=1)

# 步骤4: 将计算结果直接赋值给DataFrame（避免逐行操作）
features_df['minimum_dis'] = distances

In [17]:
print(EC_cells_df['x_centroid'].min())
print(EC_cells_df['x_centroid'].max())

print(EC_cells_df['y_centroid'].min())
print(EC_cells_df['y_centroid'].max())

850.0692138671875
9785.61328125
14489.6416015625
16019.6279296875


In [18]:
features_df.head(100)

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance,codeword_index,codeword_category,is_gene,minimum_dis
12428735,282716222801408,lihjilbc-1,0,A1cf,993.625000,14946.546875,15.421875,28.75,V3,0.390625,3773,predesigned_gene,True,45.039639
12428736,282716222814362,lkkhpjpk-1,1,A1cf,948.890625,14993.984375,24.453125,40.00,V3,0.000000,3773,predesigned_gene,True,1.276320
12428737,282716222816460,lihkagjm-1,0,A1cf,985.125000,14932.906250,17.218750,40.00,V3,0.109375,3773,predesigned_gene,True,60.917739
12428738,282716224519604,lifffeip-1,1,A1cf,866.046875,14803.796875,16.703125,40.00,V3,0.000000,3773,predesigned_gene,True,1.582286
12428739,282716224520901,ligfpmbp-1,1,A1cf,885.437500,14950.171875,16.765625,28.75,V3,0.000000,3773,predesigned_gene,True,66.561979
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12428834,282716223954253,lihhdcgf-1,0,Aatf,937.531250,14967.031250,21.625000,30.25,V3,2.546875,6462,predesigned_gene,True,18.317360
12428835,282716223954363,lifnacmm-1,0,Aatf,939.000000,14978.515625,19.640625,26.25,V3,0.640625,6462,predesigned_gene,True,7.259615
12428836,282716223954695,lihgmkoe-1,1,Aatf,943.671875,14981.171875,22.015625,28.25,V3,0.000000,6462,predesigned_gene,True,3.326567
12428837,282716223955090,lihgmoko-1,0,Aatf,948.468750,14969.171875,18.203125,40.00,V3,0.250000,6462,predesigned_gene,True,16.181172


In [19]:
# features_df.to_csv('distance.csv', index = False)

In [20]:
features_df[features_df['feature_name'] == 'Bmp7'].head(30)

,transcript_id,cell_id,overlaps_nucleus,feature_name,x_location,y_location,z_location,qv,fov_name,nucleus_distance,codeword_index,codeword_category,is_gene,minimum_dis
12463629,282716222835524,lifokkii-1,1,Bmp7,940.078125,14991.578125,28.984375,40.00,V3,0.000000,4632,predesigned_gene,True,2.228094
12463630,282716223117673,ligdlhha-1,1,Bmp7,876.640625,14977.250000,20.062500,40.00,V3,0.000000,4632,predesigned_gene,True,45.858551
12463631,282716223119080,ligcakcb-1,1,Bmp7,895.531250,14991.734375,19.796875,28.25,V3,0.000000,4632,predesigned_gene,True,35.020694
12463632,282716223120317,ligbhmph-1,1,Bmp7,920.765625,14988.703125,17.765625,24.50,V3,0.000000,4632,predesigned_gene,True,16.485375
12463633,282716223985537,ligddcij-1,1,Bmp7,854.828125,14992.625000,26.515625,40.00,V3,0.000000,4632,predesigned_gene,True,34.599788
12463634,282716222423605,ligeomjd-1,0,Bmp7,850.265625,14929.218750,22.625000,40.00,V3,0.671875,10896,predesigned_gene,True,96.214934
12463635,282716222427850,lkjhdede-1,0,Bmp7,933.984375,14916.406250,25.062500,40.00,V3,1.562500,10896,predesigned_gene,True,68.670480
12463636,282716223506539,lkjhdede-1,0,Bmp7,935.687500,14915.875000,19.875000,40.00,V3,1.234375,10896,predesigned_gene,True,68.991631
12777533,282716222831094,ligaoagl-1,1,Bmp7,874.375000,14995.937500,23.765625,15.75,V3,0.000000,4632,predesigned_gene,True,27.400037
12777534,282716223122946,lihgfpbb-1,1,Bmp7,949.296875,14973.156250,22.171875,10.75,V3,0.000000,4632,predesigned_gene,True,12.858649


In [21]:
def cutoff_pct(group):
    count_cutoff = ((group > 5).astype(int) + (group < 15 ).astype(int) == 2).sum().astype(int)
    # The 'group' argument is a DataFrame representing the current group
    return count_cutoff
res = features_df.groupby('feature_name').agg(total_counts = ('minimum_dis', 'size'),
                                       counts_cutoff = ('minimum_dis', cutoff_pct))

In [22]:
res['pct'] = res['counts_cutoff'].div(res['total_counts']) 

In [23]:
res = res.sort_values(by='pct', ascending=False)

In [24]:
res.head(10)

,total_counts,counts_cutoff,pct
feature_name,,,
Col4a5,10290,5777,0.561419
Lmo3,6001,3339,0.556407
Mmp13,948,521,0.549578
Sall1,4515,2457,0.544186
Smoc1,11407,6110,0.535636
Bmp3,10805,5524,0.511245
Nrg1,7789,3899,0.500578
Bmp5,6949,3458,0.497626
Bdkrb2,5230,2583,0.493881


In [25]:
res.to_csv('pct-of-all-features.csv', index = True)